# Notebook 3: Chunk Boundary Analysis

**Goal:** Measure information loss at chunk boundaries across strategies.  
**Key metric:** *Information completeness* — what fraction of ground-truth answer tokens are
contained within a single chunk? A low score means the answer is split across chunks,
requiring perfect multi-hop retrieval.

**Output:** `chunk_completeness.json` — completeness scores per chunking strategy per query

In [ ]:
import sys
sys.path.insert(0, '..')

import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from analysis.utils import load_results, results_to_query_df, apply_plot_style, save_output
from rag.config_loader import build_embeddings, load_documents, chunk_documents
apply_plot_style()

In [ ]:
import json
from pathlib import Path

dataset = [json.loads(l) for l in Path('experiments/datasets/fmcg_product_qa.jsonl').read_text().splitlines() if l.strip()]
print(f'Dataset size: {len(dataset)} queries')

## 1. Build Chunk Sets for Each Strategy

In [ ]:
embedding_cfg = {'provider': 'azure_openai', 'model': 'text-embedding-3-small'}
embeddings_model = build_embeddings(embedding_cfg)
docs = load_documents('data/fmcg_docs')

STRATEGIES = [
    ('fixed',        {'strategy': 'fixed',        'chunk_size': 128, 'overlap': 20}),
    ('semantic',     {'strategy': 'semantic',      'chunk_size': 128, 'overlap': 20}),
    ('parent_child', {'strategy': 'parent_child',  'chunk_size': 128, 'overlap': 20}),
]

strategy_chunks = {}
for name, cfg in STRATEGIES:
    chunks = chunk_documents(docs, cfg, embeddings_model)
    strategy_chunks[name] = [c.page_content for c in chunks]
    print(f'{name}: {len(chunks)} chunks')

## 2. Compute Information Completeness per Query

In [ ]:
def token_set(text: str) -> set[str]:
    return set(re.findall(r'\w+', text.lower()))

def max_single_chunk_coverage(ground_truth: str, chunks: list[str]) -> float:
    """Maximum fraction of ground-truth tokens found in any single chunk."""
    gt_tokens = token_set(ground_truth)
    if not gt_tokens:
        return 0.0
    best = max(len(gt_tokens & token_set(c)) / len(gt_tokens) for c in chunks)
    return best

rows = []
for row in dataset:
    for strategy, chunks in strategy_chunks.items():
        completeness = max_single_chunk_coverage(row['ground_truth'], chunks)
        rows.append({
            'query_id': row.get('query_id', ''),
            'query': row['query'],
            'strategy': strategy,
            'completeness': completeness,
        })

completeness_df = pd.DataFrame(rows)
completeness_df.groupby('strategy')['completeness'].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution plot
for strategy in completeness_df['strategy'].unique():
    data = completeness_df[completeness_df['strategy'] == strategy]['completeness']
    axes[0].hist(data, bins=20, alpha=0.6, label=strategy, density=True)
axes[0].set_xlabel('Max Single-Chunk Coverage')
axes[0].set_ylabel('Density')
axes[0].set_title('Information Completeness Distribution')
axes[0].legend()

# Mean completeness by strategy
mean_comp = completeness_df.groupby('strategy')['completeness'].mean().sort_values(ascending=False)
mean_comp.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Mean Completeness by Strategy')
axes[1].set_ylabel('Mean completeness score')
axes[1].set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Split-Fact Queries — Where No Single Chunk Contains the Answer

In [ ]:
# Queries where best fixed-strategy coverage < 40%
fixed_completeness = completeness_df[completeness_df['strategy'] == 'fixed']
split_queries = fixed_completeness[fixed_completeness['completeness'] < 0.4]

print(f'Split-fact queries (fixed chunking coverage < 40%): {len(split_queries)}')
for _, row in split_queries.iterrows():
    print(f"  [{row['completeness']:.2f}] {row['query']}")

In [ ]:
save_output(completeness_df, 'chunk_completeness.json')
print('Saved.')